# KADMON Optuna — Binning + Partial OT

Le classement utilise `global_distance_mm`. Pour chaque bundle, l'objectif Optuna minimise le rapport entre la distance intra-identité et la distance moyenne inter-identité.


## 1. Imports et configuration


In [1]:
from pathlib import Path
import sys
from time import perf_counter

import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from plotly.io import show


NOTEBOOK_DIR = Path.cwd().resolve()
KADMON_ROOT = (NOTEBOOK_DIR / "../..").resolve()
BUNDLES_DIR = NOTEBOOK_DIR.parent / "bundles"
STUDY_PATH = NOTEBOOK_DIR / "studies" / "optuna_reid.sqlite3"
if str(KADMON_ROOT) not in sys.path:
    sys.path.insert(0, str(KADMON_ROOT))

from kadmon.comparison import compare_bundles
from kadmon.io import BundleCollection
from kadmon.protocol import HCP_REID_PROTOCOL
from kadmon.reid import aggregate_reid_metrics, bundle_reid_metrics, set_trial_metrics
from kadmon.selection import select_best_reid_trial
from kadmon.optimization import StudyRunPolicy, create_reid_study, run_until_complete

optuna.logging.set_verbosity(optuna.logging.WARNING)

EXPERIMENT_NAME = "binning_partial"
COMPRESSION = "binning"
TRANSPORT = "partial"
N_TRIALS = 120


Info: some functions in tractosearch.resampling are faster when 'numba' is installed


## 2. Données

Les bundles sont chargés depuis le dossier frère `../bundles`.


In [2]:
REFERENCE_SUBJECT = HCP_REID_PROTOCOL.reference_subject
INTRA_IDENTITY_SUBJECT = HCP_REID_PROTOCOL.intra_identity_subject
COMPARISON_SUBJECTS = HCP_REID_PROTOCOL.comparison_subjects
SUBJECTS = HCP_REID_PROTOCOL.subjects
N_POINTS = HCP_REID_PROTOCOL.n_points
SEED = 42
BUNDLES_TO_RUN = None  # tuple de noms exacts pour un essai court, sinon None
# Partial OT construit des matrices denses : rester séquentiel évite de
# multiplier leur mémoire par le nombre de workers.
N_JOBS_BUNDLES = 1
MAX_COST_MATRIX_BYTES = 512 * 2**20  # 512 Mio estimés pour la comparaison dense

bundles = BundleCollection(
    BUNDLES_DIR, SUBJECTS, n_points=N_POINTS, selected=BUNDLES_TO_RUN
)
SUBJECT_FILES = bundles.files
bundle_names = bundles.names
bundle_cache = bundles.cache
# Le cache inclut aussi le type et tous les paramètres de compression via kadmon.
compression_cache = {}


load_bundle = bundles.load


print(f"Données : {BUNDLES_DIR}")
print("Protocole : 1 comparaison intra-identité et 9 comparaisons inter-identité par bundle")
print(f"Bundles communs : {len(bundle_names)}")
display(pd.DataFrame({"bundle": bundle_names}))


Données : /home/colin/Tractographie/KADMON/notebooks/bundles
Protocole : 1 comparaison intra-identité et 9 comparaisons inter-identité par bundle
Bundles communs : 31


,bundle
0,tractosearch_nn_8_0mm_all_AF_L_m
1,tractosearch_nn_8_0mm_all_AF_R_m
2,tractosearch_nn_8_0mm_all_CC_1_m
3,tractosearch_nn_8_0mm_all_CC_2a_m
4,tractosearch_nn_8_0mm_all_CC_2b_m
5,tractosearch_nn_8_0mm_all_CC_3_m
6,tractosearch_nn_8_0mm_all_CC_4_m
7,tractosearch_nn_8_0mm_all_CC_5_m
8,tractosearch_nn_8_0mm_all_CC_6_m
9,tractosearch_nn_8_0mm_all_CC_7_m


## 3. Espace de recherche

- `bin_size` : 2 à 16 mm, par pas de 1 mm
- `binning_nb` : 2 ou 3 (seules valeurs implémentées par tractosearch)
- `method` : `mean` ou `median`
- `mass` : 0,50 à 1,00, par pas de 0,01

`n_points=12` reste fixe.

> **Note sur les points réservoirs (`nb_dummies=100`).** Partial OT ajoute des
> points artificiels pour recevoir la fraction de masse qui n'est pas transportée
> (par exemple 16 % si `mass=0.84`). Le binning peut produire des milliers de
> représentants aux poids très petits et irréguliers; concentrer toute cette masse
> sur l'unique dummy utilisé par défaut dans POT peut alors déstabiliser le solveur
> EMD. La répartir sur 100 dummies améliore sa stabilité. Ce problème apparaît
> beaucoup moins avec K-means, qui impose peu de centroïdes aux poids plus élevés.
> Ces points sont purement numériques et sont retirés du plan avant le calcul des
> métriques : ils ne représentent pas de vraies fibres.


In [3]:
def sample_parameters(trial):
    return (
        {
            "bin_size": trial.suggest_float("bin_size", 2.0, 16.0, step=1.0),
            "binning_nb": trial.suggest_int("binning_nb", 2, 3),
            "method": trial.suggest_categorical("method", ["mean", "median"]),
            "n_points": N_POINTS,
        },
        {
            "mass": trial.suggest_float("mass", 0.50, 1.00, step=0.01),
            "nb_dummies": 100,
        },
    )


## 4. Objectif Optuna

La compression, MDF, le transport et les statistiques sont calculés par `compare_bundles()`.


In [4]:
def evaluate_bundle(bundle_name, compression_parameters, transport_parameters):
    source = load_bundle(REFERENCE_SUBJECT, bundle_name)

    pair_metrics = []
    for candidate_subject in COMPARISON_SUBJECTS:
        target = load_bundle(candidate_subject, bundle_name)
        try:
            result = compare_bundles(
                source,
                target,
                compression=COMPRESSION,
                transport=TRANSPORT,
                compression_parameters=compression_parameters,
                transport_parameters=transport_parameters,
                compression_cache=compression_cache,
                source_compression_key=(REFERENCE_SUBJECT, bundle_name),
                target_compression_key=(candidate_subject, bundle_name),
                max_cost_matrix_bytes=MAX_COST_MATRIX_BYTES,
            )
        except MemoryError as exc:
            raise optuna.TrialPruned(f"Configuration trop volumineuse : {exc}") from exc
        except ValueError as exc:
            if "Error in the EMD resolution" in str(exc):
                raise optuna.TrialPruned(f"EMD instable : {exc}") from exc
            raise
        metrics = result["metrics"]
        pair_metrics.append({
            "global_distance_mm": float(metrics["global_distance_mm"]),
            "mean_displacement_mm": float(metrics["mean_mm"]),
            "transported_mass": float(metrics["transported_mass"]),
            "source_n_representatives": int(metrics["source_n_representatives"]),
            "target_n_representatives": int(metrics["target_n_representatives"]),
        })

    # La compression de la référence doit être reproductible pour tous les candidats.
    if len({row["source_n_representatives"] for row in pair_metrics}) != 1:
        raise RuntimeError(f"Compression source non reproductible pour {bundle_name}.")

    return bundle_reid_metrics(
        bundle_name, [row["global_distance_mm"] for row in pair_metrics],
        comparison_subjects=COMPARISON_SUBJECTS,
        intra_identity_subject=INTRA_IDENTITY_SUBJECT,
        mean_displacement_mm=np.mean([row["mean_displacement_mm"] for row in pair_metrics]),
        mean_transported_mass=np.mean([row["transported_mass"] for row in pair_metrics]),
        mean_n_representatives=np.mean([value for row in pair_metrics for value in (row["source_n_representatives"], row["target_n_representatives"])]),
    )


def objective(trial):
    started = perf_counter()
    compression_parameters, transport_parameters = sample_parameters(trial)
    tasks = (
        delayed(evaluate_bundle)(
            bundle_name, compression_parameters, transport_parameters
        )
        for bundle_name in bundle_names
    )
    results = (
        [
            evaluate_bundle(
                bundle_name, compression_parameters, transport_parameters
            )
            for bundle_name in bundle_names
        ]
        if N_JOBS_BUNDLES == 1
        else Parallel(
            n_jobs=min(N_JOBS_BUNDLES, len(bundle_names)), backend="threading"
        )(tasks)
    )
    bundle_metrics = pd.DataFrame(results)
    aggregates = aggregate_reid_metrics(
        bundle_metrics, n_comparison_subjects=len(COMPARISON_SUBJECTS),
        elapsed_s=perf_counter() - started,
    )
    set_trial_metrics(trial, aggregates)
    return aggregates["mean_intra_inter_ratio"]


## 5. Optimisation

La base SQLite est l'unique sortie automatique de l'étude.


In [5]:
study = create_reid_study(STUDY_PATH, EXPERIMENT_NAME, seed=SEED)
run_until_complete(
    study, objective, StudyRunPolicy(N_TRIALS),
    callbacks=[lambda study, trial: compression_cache.clear()],
)

print(f"Base SQLite : {STUDY_PATH}")


Étude : 120/120 essais COMPLETE; cible restante=0.
Base SQLite : /home/colin/Tractographie/KADMON/notebooks/optuna/studies/optuna_reid.sqlite3


## 6. Analyse des résultats


In [6]:
trials_df = study.trials_dataframe(
    attrs=("number", "value", "params", "user_attrs", "state")
)
display(trials_df.sort_values("value").head(10))

ratio_best = study.best_trial
reid_best = select_best_reid_trial(study.trials)
best_summary = pd.Series({
    "experiment": EXPERIMENT_NAME,
    "trial": reid_best.number,
    "selection": "Top-1, rang, ratio, couverture",
    "mean_intra_inter_ratio": reid_best.value,
    **reid_best.params,
    **reid_best.user_attrs,
}, name="meilleur essai RE-ID")
display(best_summary.to_frame())
print(f"Optimum brut du ratio : trial {ratio_best.number}")


,number,value,params_bin_size,params_binning_nb,params_mass,params_method,user_attrs_elapsed_s,user_attrs_intra_identity_top1_accuracy,user_attrs_mean_displacement_mm,user_attrs_mean_inter_identity_distance_mm,...,user_attrs_mean_intra_inter_ratio,user_attrs_mean_intra_inter_separation_margin_mm,user_attrs_mean_n_representatives,user_attrs_mean_transported_mass,user_attrs_median_intra_inter_ratio,user_attrs_n_bundles,user_attrs_n_comparisons,user_attrs_reid_failed_bundles,user_attrs_reid_valid_bundles,state
108,108,0.314944,16.0,2,0.5,mean,591.916731,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
109,109,0.314944,16.0,2,0.5,mean,590.925114,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
120,120,0.314944,16.0,2,0.5,mean,595.912299,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
112,112,0.314944,16.0,2,0.5,mean,595.657399,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
118,118,0.314944,16.0,2,0.5,mean,592.255895,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
100,100,0.314944,16.0,2,0.5,mean,594.499368,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
89,89,0.314944,16.0,2,0.5,mean,594.237078,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
140,140,0.314944,16.0,2,0.5,mean,615.804627,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
137,137,0.314944,16.0,2,0.5,mean,594.928198,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
138,138,0.314944,16.0,2,0.5,mean,593.313063,0.967742,3.589178,4.075891,...,0.314944,1.392718,146.562903,0.5,0.270595,31.0,310.0,[tractosearch_nn_8_0mm_all_CST_L_m],"[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE


,meilleur essai RE-ID
experiment,binning_partial
trial,121
selection,"Top-1, rang, ratio, couverture"
mean_intra_inter_ratio,0.386496
bin_size,8.0
binning_nb,2
method,mean
mass,0.5
elapsed_s,894.620375
intra_identity_top1_accuracy,1.0


Optimum brut du ratio : trial 89


## 7. Visualisations Optuna

Les importances donnent d'abord une vue globale, puis les coupes montrent l'effet individuel de chaque paramètre. Le contour est limité à `bin_size` et `mass`, les deux paramètres continus pour lesquels une surface 2D est réellement lisible.


In [7]:
fig_importance = optuna.visualization.plot_param_importances(study)
show(fig_importance)


In [8]:
df = study.trials_dataframe()[[
    "number", "value", "state", "params_mass", "params_bin_size",
    "params_binning_nb", "params_method",
    "user_attrs_mean_n_representatives",
    "user_attrs_intra_identity_top1_accuracy",
]]
df = df[df["state"] == "COMPLETE"].dropna(subset=["value"]).rename(columns={
    "number": "trial", "value": "score",
    "params_mass": "mass", "params_bin_size": "bin_size",
    "params_binning_nb": "binning_nb", "params_method": "method",
    "user_attrs_mean_n_representatives": "n_representatives",
    "user_attrs_intra_identity_top1_accuracy": "reid_accuracy",
})
best_score = df["score"].min()

rows, selected_trials = [], set()
for threshold in (1, 2, 5):
    candidates = df[
        (df["score"] <= best_score * (1 + threshold / 100))
        & ~df["trial"].isin(selected_trials)
    ]
    # Ne pas répéter un trial déjà choisi; parmi les solutions restantes,
    # maximiser mass conserve davantage du faisceau.
    row = candidates.sort_values(
        ["mass", "score"], ascending=[False, True]
    ).iloc[0]
    selected_trials.add(int(row.trial))
    rows.append([
        threshold, int(row.trial), row.score,
        100 * (row.score / best_score - 1), row.mass, row.bin_size,
        int(row.binning_nb), row.method, row.n_representatives,
        row.reid_accuracy,
    ])

tradeoff = pd.DataFrame(rows, columns=[
    "seuil (%)", "trial", "score", "écart relatif (%)", "mass",
    "bin_size", "binning_nb", "method", "n_representatives",
    "reid_accuracy",
])
display(tradeoff.style.format({
    "score": "{:.6f}", "écart relatif (%)": "{:.2f}",
    "mass": "{:.2f}", "bin_size": "{:.0f}",
    "n_representatives": "{:.0f}", "reid_accuracy": "{:.0%}",
}))


,seuil (%),trial,score,écart relatif (%),mass,bin_size,binning_nb,method,n_representatives,reid_accuracy
0,1,55,0.317031,0.66,0.68,16,2,mean,147,97%
1,2,56,0.316408,0.47,0.61,16,2,mean,147,97%
2,5,57,0.316272,0.42,0.60,16,2,mean,147,97%


## 8. Interprétation des compromis observés

L'étude finale contient **120 essais `COMPLETE`**. Elle compte aussi 14 essais `PRUNED`, 5 essais `FAIL` et 2 anciens essais `RUNNING`; seuls les essais `COMPLETE` sont utilisés pour le classement.

Le **trial 89** est l'optimum strict observé (`score=0,314944`, `bin_size=16 mm`, `binning_nb=2`, `method=mean`, `mass=0,50`). La compression produit en moyenne **147 représentants** par distribution. Son exactitude RE-ID Top-1 est de **96,77 %** (30 bundles sur 31); le seul échec est `CST_L`.

Le **trial 55** constitue le meilleur compromis proche de l'optimum : `score=0,317031`, `bin_size=16 mm`, `binning_nb=2`, `method=mean`, `mass=0,68`, soit seulement **+0,66 %** sur le score. Il conserve 36 % de masse transportée supplémentaire, avec la même exactitude Top-1 de 96,77 % et le même échec `CST_L`. Le tableau 1–2–5 % exclut maintenant les trials déjà sélectionnés afin de présenter trois alternatives distinctes. Dans cette étude, même les alternatives affichées aux seuils 2 % et 5 % se trouvent déjà à moins de 1 % de l'optimum; il n'existe aucun essai `COMPLETE` dans la bande 2–5 %.

- Selon la règle RE-ID commune, le **trial 121** est le meilleur classement (`bin_size=8 mm`, `binning_nb=2`, `method=mean`, `mass=0,50`) : Top-1 100 %, rang moyen 1,00 et environ 996 représentants.
- Pour l'usage anatomique de KADMON, le projet utilise le **trial 55 par défaut** : `mass=0,68`, même Top-1 que l'optimum strict local à 16 mm et couverture supérieure.
- Pour une future initialisation **LDDMM**, privilégier également le trial 55, car il transporte une fraction plus importante du faisceau sans perte d'exactitude Top-1.
- L'optimum de `bin_size` se trouve à la borne supérieure testée (16 mm). Cela indique qu'une compression plus forte stabilise la RE-ID dans cet espace, mais justifie un contrôle ciblé au-delà de 16 mm avant de conclure que 16 mm est l'optimum réel.
- Contrairement à `epsilon` dans Sinkhorn, `mass` représente ici une **couverture anatomique explicite** : une masse faible améliore parfois le score en ignorant davantage de streamlines, tandis qu'une masse élevée produit une comparaison plus complète du faisceau.
